# Day 25 — Point-in-Time Economic Data

## Objectives
- Understand economic observation dates versus release dates
- Retrieve historical economic data vintages
- Build a point-in-time economic database
- Prevent look-ahead bias in historical investment strategies

In [1]:

import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

conn = sqlite3.connect("hedge_fund.db")

conn.execute("""
CREATE TABLE IF NOT EXISTS economic_vintages (
    indicator TEXT NOT NULL,
    observation_date TEXT NOT NULL,
    available_date TEXT NOT NULL,
    vintage_end_date TEXT,
    value REAL NOT NULL,
    source TEXT NOT NULL,
    PRIMARY KEY (
        indicator,
        observation_date,
        available_date
    )
)
""")

conn.execute("""
CREATE INDEX IF NOT EXISTS idx_economic_vintages_available
ON economic_vintages(indicator, available_date)
""")

conn.commit()

print("Economic vintage table created successfully.")

Economic vintage table created successfully.


In [2]:

economic_summary = pd.read_sql_query("""
SELECT
    indicator,
    COUNT(*) AS observations,
    MIN(observation_date) AS first_observation,
    MAX(observation_date) AS latest_observation
FROM economic_data
GROUP BY indicator
ORDER BY indicator
""", conn)

display(economic_summary)

,indicator,observations,first_observation,latest_observation
0,CPI,139,2015-01-01,2026-08-01
1,FED_RATE,140,2015-01-01,2026-08-01
2,REAL_GDP,46,2015-01-01,2026-04-01
3,UNEMPLOYMENT,139,2015-01-01,2026-08-01


In [4]:

import os
from dotenv import load_dotenv

load_dotenv(".env")

api_key = os.getenv("FRED_API_KEY")

if api_key:
    print("FRED API key loaded successfully!")
else:
    print("API key not found. Check your .env file.")

FRED API key loaded successfully!


In [5]:

import requests
import pandas as pd

# Retrieve the CPI data available on June 30, 2025.
url = "https://api.stlouisfed.org/fred/series/observations"

params = {
    "series_id": "CPIAUCSL",
    "api_key": api_key,
    "file_type": "json",
    "realtime_start": "2025-06-30",
    "realtime_end": "2025-06-30",
    "observation_start": "2025-01-01",
    "observation_end": "2025-06-30"
}

response = requests.get(
    url,
    params=params,
    timeout=30
)

response.raise_for_status()

cpi_vintage = pd.DataFrame(
    response.json()["observations"]
)

display(
    cpi_vintage[
        ["date", "value", "realtime_start", "realtime_end"]
    ]
)

,date,value,realtime_start,realtime_end
0,2025-01-01,319.086,2025-06-30,2025-06-30
1,2025-02-01,319.775,2025-06-30,2025-06-30
2,2025-03-01,319.615,2025-06-30,2025-06-30
3,2025-04-01,320.321,2025-06-30,2025-06-30
4,2025-05-01,320.580,2025-06-30,2025-06-30


In [6]:

# Retrieve historical CPI vintages for a manageable period

vintage_params = {
    "series_id": "CPIAUCSL",
    "api_key": api_key,
    "file_type": "json",
    "realtime_start": "2025-01-01",
    "realtime_end": "2025-12-31",
    "observation_start": "2025-01-01",
    "observation_end": "2025-06-30",
    "output_type": 2
}

response = requests.get(
    "https://api.stlouisfed.org/fred/series/observations",
    params=vintage_params,
    timeout=60
)

response.raise_for_status()

cpi_history = pd.DataFrame(
    response.json()["observations"]
)

display(cpi_history.head(10))

print("Historical vintage records:", len(cpi_history))

,date,CPIAUCSL_20250212,CPIAUCSL_20250312,CPIAUCSL_20250410,CPIAUCSL_20250513,CPIAUCSL_20250611,CPIAUCSL_20250715,CPIAUCSL_20250812,CPIAUCSL_20250911,CPIAUCSL_20251024,CPIAUCSL_20251218,CPIAUCSL_20251231
0,2025-01-01,319.086,319.086,319.086,319.086,319.086,319.086,319.086,319.086,319.086,319.086,319.086
1,2025-02-01,NaN,319.775,319.775,319.775,319.775,319.775,319.775,319.775,319.775,319.775,319.775
2,2025-03-01,NaN,NaN,319.615,319.615,319.615,319.615,319.615,319.615,319.615,319.615,319.615
3,2025-04-01,NaN,NaN,NaN,320.321,320.321,320.321,320.321,320.321,320.321,320.321,320.321
4,2025-05-01,NaN,NaN,NaN,NaN,320.580,320.580,320.580,320.580,320.580,320.580,320.580
5,2025-06-01,NaN,NaN,NaN,NaN,NaN,321.500,321.500,321.500,321.500,321.500,321.500


Historical vintage records: 6


In [9]:

# Day 25 — Retrieve CPI observations with vintage intervals

vintage_params = {
    "series_id": "CPIAUCSL",
    "api_key": api_key,
    "file_type": "json",
    "realtime_start": "2025-01-01",
    "realtime_end": "2025-12-31",
    "observation_start": "2025-01-01",
    "observation_end": "2025-06-30",
    "output_type": 1
}

response = requests.get(
    "https://api.stlouisfed.org/fred/series/observations",
    params=vintage_params,
    timeout=60
)

response.raise_for_status()

cpi_history = pd.DataFrame(
    response.json()["observations"]
)

required_columns = {
    "date",
    "value",
    "realtime_start",
    "realtime_end"
}

missing = required_columns - set(cpi_history.columns)

if missing:
    raise ValueError(f"Missing API columns: {missing}")

print("Records retrieved:", len(cpi_history))
display(cpi_history.head(10))

Records retrieved: 6


,realtime_start,realtime_end,date,value
0,2025-02-12,2025-12-31,2025-01-01,319.086
1,2025-03-12,2025-12-31,2025-02-01,319.775
2,2025-04-10,2025-12-31,2025-03-01,319.615
3,2025-05-13,2025-12-31,2025-04-01,320.321
4,2025-06-11,2025-12-31,2025-05-01,320.580
5,2025-07-15,2025-12-31,2025-06-01,321.500


In [10]:

cpi_records = cpi_history.rename(columns={
    "date": "observation_date",
    "realtime_start": "available_date",
    "realtime_end": "vintage_end_date"
}).copy()

cpi_records["indicator"] = "CPI"
cpi_records["source"] = "ALFRED"

cpi_records["value"] = pd.to_numeric(
    cpi_records["value"],
    errors="coerce"
)

cpi_records = cpi_records.dropna(subset=["value"])

columns = [
    "indicator",
    "observation_date",
    "available_date",
    "vintage_end_date",
    "value",
    "source"
]

conn.executemany("""
    INSERT OR REPLACE INTO economic_vintages (
        indicator,
        observation_date,
        available_date,
        vintage_end_date,
        value,
        source
    )
    VALUES (?, ?, ?, ?, ?, ?)
""", cpi_records[columns].itertuples(
    index=False,
    name=None
))

conn.commit()

print("CPI vintage records saved:", len(cpi_records))

display(pd.read_sql_query("""
    SELECT *
    FROM economic_vintages
    WHERE indicator = 'CPI'
    ORDER BY observation_date, available_date
    LIMIT 15
""", conn))

CPI vintage records saved: 6


,indicator,observation_date,available_date,vintage_end_date,value,source
0,CPI,2025-01-01,2025-02-12,2025-12-31,319.086,ALFRED
1,CPI,2025-02-01,2025-03-12,2025-12-31,319.775,ALFRED
2,CPI,2025-03-01,2025-04-10,2025-12-31,319.615,ALFRED
3,CPI,2025-04-01,2025-05-13,2025-12-31,320.321,ALFRED
4,CPI,2025-05-01,2025-06-11,2025-12-31,320.580,ALFRED
5,CPI,2025-06-01,2025-07-15,2025-12-31,321.500,ALFRED


In [11]:

# Day 25 — Historical information cutoff test

as_of_date = "2025-06-30"

point_in_time_cpi = pd.read_sql_query("""
WITH ranked AS (
    SELECT
        observation_date,
        value,
        available_date,
        ROW_NUMBER() OVER (
            PARTITION BY observation_date
            ORDER BY available_date DESC
        ) AS rn
    FROM economic_vintages
    WHERE indicator = 'CPI'
      AND available_date <= ?
      AND (
          vintage_end_date IS NULL
          OR vintage_end_date >= ?
      )
)
SELECT
    observation_date,
    value,
    available_date
FROM ranked
WHERE rn = 1
ORDER BY observation_date
""", conn, params=(as_of_date, as_of_date))

print("CPI observations available by:", as_of_date)
display(point_in_time_cpi)

# Verify that no future information was included.
assert (
    point_in_time_cpi["available_date"] <= as_of_date
).all()

assert "2025-06-01" not in (
    point_in_time_cpi["observation_date"].tolist()
)

print("PASS: No observations released after the cutoff.")

CPI observations available by: 2025-06-30


,observation_date,value,available_date
0,2025-01-01,319.086,2025-02-12
1,2025-02-01,319.775,2025-03-12
2,2025-03-01,319.615,2025-04-10
3,2025-04-01,320.321,2025-05-13
4,2025-05-01,320.580,2025-06-11


PASS: No observations released after the cutoff.


In [12]:

# Inspect ALFRED's vintage-oriented response

revision_params = {
    "series_id": "CPIAUCSL",
    "api_key": api_key,
    "file_type": "json",
    "realtime_start": "2025-01-01",
    "realtime_end": "2026-09-24",
    "observation_start": "2025-01-01",
    "observation_end": "2025-06-30",
    "output_type": 2
}

revision_response = requests.get(
    "https://api.stlouisfed.org/fred/series/observations",
    params=revision_params,
    timeout=60
)

revision_response.raise_for_status()

revision_data = revision_response.json()

print("Response fields:")
print(list(revision_data.keys()))

print("\nFirst two records:")
display(pd.DataFrame(
    revision_data["observations"]
).head(2))

Response fields:
['realtime_start', 'realtime_end', 'observation_start', 'observation_end', 'units', 'output_type', 'file_type', 'order_by', 'sort_order', 'count', 'offset', 'limit', 'observations']

First two records:


,date,CPIAUCSL_20250212,CPIAUCSL_20250312,CPIAUCSL_20250410,CPIAUCSL_20250513,CPIAUCSL_20250611,CPIAUCSL_20250715,CPIAUCSL_20250812,CPIAUCSL_20250911,CPIAUCSL_20251024,...,CPIAUCSL_20260113,CPIAUCSL_20260213,CPIAUCSL_20260311,CPIAUCSL_20260410,CPIAUCSL_20260512,CPIAUCSL_20260610,CPIAUCSL_20260714,CPIAUCSL_20260812,CPIAUCSL_20260911,CPIAUCSL_20260924
0,2025-01-01,319.086,319.086,319.086,319.086,319.086,319.086,319.086,319.086,319.086,...,319.086,318.961,318.961,318.961,318.961,318.961,318.961,318.961,318.961,318.961
1,2025-02-01,NaN,319.775,319.775,319.775,319.775,319.775,319.775,319.775,319.775,...,319.775,319.679,319.679,319.679,319.679,319.679,319.679,319.679,319.679,319.679


In [13]:

# Convert ALFRED's wide vintage table into long format

wide = pd.DataFrame(revision_data["observations"])

long = wide.melt(
    id_vars=["date"],
    var_name="vintage_column",
    value_name="value"
)

# Example: CPIAUCSL_20250212 -> 2025-02-12
long["available_date"] = pd.to_datetime(
    long["vintage_column"].str.extract(
        r"_(\d{8})$"
    )[0],
    format="%Y%m%d",
    errors="coerce"
)

long["value"] = pd.to_numeric(
    long["value"],
    errors="coerce"
)

long = long.dropna(
    subset=["available_date", "value"]
).copy()

long["observation_date"] = pd.to_datetime(
    long["date"]
)

long = long.sort_values(
    ["observation_date", "available_date"]
)

display(long.head(10))
print("Vintage records:", len(long))

,date,vintage_column,value,available_date,observation_date
0,2025-01-01,CPIAUCSL_20250212,319.086,2025-02-12,2025-01-01
6,2025-01-01,CPIAUCSL_20250312,319.086,2025-03-12,2025-01-01
12,2025-01-01,CPIAUCSL_20250410,319.086,2025-04-10,2025-01-01
18,2025-01-01,CPIAUCSL_20250513,319.086,2025-05-13,2025-01-01
24,2025-01-01,CPIAUCSL_20250611,319.086,2025-06-11,2025-01-01
30,2025-01-01,CPIAUCSL_20250715,319.086,2025-07-15,2025-01-01
36,2025-01-01,CPIAUCSL_20250812,319.086,2025-08-12,2025-01-01
42,2025-01-01,CPIAUCSL_20250911,319.086,2025-09-11,2025-01-01
48,2025-01-01,CPIAUCSL_20251024,319.086,2025-10-24,2025-01-01
54,2025-01-01,CPIAUCSL_20251218,319.086,2025-12-18,2025-01-01


Vintage records: 105


In [14]:

conn.execute("""
CREATE TABLE IF NOT EXISTS economic_vintage_snapshots (
    indicator TEXT NOT NULL,
    observation_date TEXT NOT NULL,
    snapshot_date TEXT NOT NULL,
    value REAL NOT NULL,
    source TEXT NOT NULL,
    PRIMARY KEY (
        indicator,
        observation_date,
        snapshot_date
    )
)
""")

records = long.copy()
records["indicator"] = "CPI"
records["source"] = "ALFRED"

records["observation_date"] = (
    records["observation_date"].dt.strftime("%Y-%m-%d")
)

records["snapshot_date"] = (
    records["available_date"].dt.strftime("%Y-%m-%d")
)

conn.executemany("""
INSERT OR REPLACE INTO economic_vintage_snapshots
(indicator, observation_date, snapshot_date, value, source)
VALUES (?, ?, ?, ?, ?)
""", records[
    [
        "indicator",
        "observation_date",
        "snapshot_date",
        "value",
        "source"
    ]
].itertuples(index=False, name=None))

conn.commit()

print("Saved snapshot records:", len(records))

display(pd.read_sql_query("""
SELECT *
FROM economic_vintage_snapshots
ORDER BY observation_date, snapshot_date
LIMIT 10
""", conn))

Saved snapshot records: 105


,indicator,observation_date,snapshot_date,value,source
0,CPI,2025-01-01,2025-02-12,319.086,ALFRED
1,CPI,2025-01-01,2025-03-12,319.086,ALFRED
2,CPI,2025-01-01,2025-04-10,319.086,ALFRED
3,CPI,2025-01-01,2025-05-13,319.086,ALFRED
4,CPI,2025-01-01,2025-06-11,319.086,ALFRED
5,CPI,2025-01-01,2025-07-15,319.086,ALFRED
6,CPI,2025-01-01,2025-08-12,319.086,ALFRED
7,CPI,2025-01-01,2025-09-11,319.086,ALFRED
8,CPI,2025-01-01,2025-10-24,319.086,ALFRED
9,CPI,2025-01-01,2025-12-18,319.086,ALFRED


In [15]:

summary = pd.read_sql_query("""
SELECT
    indicator,
    COUNT(*) AS records,
    COUNT(DISTINCT observation_date) AS observations,
    COUNT(DISTINCT snapshot_date) AS snapshots,
    MIN(snapshot_date) AS first_snapshot,
    MAX(snapshot_date) AS latest_snapshot
FROM economic_vintage_snapshots
GROUP BY indicator
""", conn)

display(summary)

assert not summary.empty
assert summary["records"].iloc[0] > 0

print("PASS: Historical CPI snapshots saved.")

,indicator,records,observations,snapshots,first_snapshot,latest_snapshot
0,CPI,105,6,20,2025-02-12,2026-09-24


PASS: Historical CPI snapshots saved.


In [16]:

# Test which CPI data was available on three historical dates

test_dates = [
    "2025-05-01",
    "2025-06-30",
    "2025-07-16"
]

for as_of_date in test_dates:
    result = pd.read_sql_query("""
        WITH ranked AS (
            SELECT
                observation_date,
                available_date,
                value,
                ROW_NUMBER() OVER (
                    PARTITION BY observation_date
                    ORDER BY available_date DESC
                ) AS rn
            FROM economic_vintages
            WHERE indicator = 'CPI'
              AND available_date <= ?
              AND (
                  vintage_end_date IS NULL
                  OR vintage_end_date >= ?
              )
        )
        SELECT
            observation_date,
            available_date,
            value
        FROM ranked
        WHERE rn = 1
        ORDER BY observation_date
    """, conn, params=(as_of_date, as_of_date))

    print(f"\nCPI available as of {as_of_date}")
    display(result)

    assert (
        result["available_date"] <= as_of_date
    ).all()

print("PASS: All historical cutoff tests passed.")


CPI available as of 2025-05-01


,observation_date,available_date,value
0,2025-01-01,2025-02-12,319.086
1,2025-02-01,2025-03-12,319.775
2,2025-03-01,2025-04-10,319.615



CPI available as of 2025-06-30


,observation_date,available_date,value
0,2025-01-01,2025-02-12,319.086
1,2025-02-01,2025-03-12,319.775
2,2025-03-01,2025-04-10,319.615
3,2025-04-01,2025-05-13,320.321
4,2025-05-01,2025-06-11,320.580



CPI available as of 2025-07-16


,observation_date,available_date,value
0,2025-01-01,2025-02-12,319.086
1,2025-02-01,2025-03-12,319.775
2,2025-03-01,2025-04-10,319.615
3,2025-04-01,2025-05-13,320.321
4,2025-05-01,2025-06-11,320.580
5,2025-06-01,2025-07-15,321.500


PASS: All historical cutoff tests passed.


## Day 25 — Results

- Connected successfully to the ALFRED API.
- Created a SQL table for economic vintage intervals.
- Retrieved historical CPI observations and snapshots.
- Stored 105 CPI snapshot records.
- Tested historical information cutoffs.
- Distinguished observation dates from availability dates.

### Limitations

The current CPI extract covers six observation months.
The database is not yet a complete economic revision history.

ALFRED vintage dates do not provide intraday release
timestamps. Future trading simulations will conservatively
use the next trading session after an availability date.

Additional economic indicators and comprehensive historical
coverage are required before production-quality backtesting.